# Autograd: Automatic Differentiation

Central to all neural networks in PyTorch is the `autograd` package.  Let's briefly visit this, and we will then go to training our first neural network.

The `autograd` package provides automatic differentiation for all operations on Tensors.  It is a define-by-run framework, which means that your backprop is defined by how your code is run, and that every single iteration can be different.

Let us see this in more simple terms with some examples.

## Tensor

`torch.tensor` is the central class of the package.  If you set its attribute `.requires_grad` as `True`, it starts to track all operations on it.  When you finish your computation you can call `.backward()` and have all the gradients computed automatically.  The gradient for this tensor will be accumulated into the `.grad` attribute.

To stop a tensor from tracking history (and using memory), you can also wrap the code block in `with torch.no_grad():`.  This can be particularly helpful when evaluating a model because the model may have trainable parameters with `requires_grad=True`, but for which we don't need the gradients.

There's one more class which is very important for autograd implementation - a `Function`.

`Tensor` and `Function` are interconnected and build up an acyclic graph, that encodes a complete history of computation.  Each tensor has a `.grad_fn` attribute that references a `Function` that has created the `Tensor` (except for Tensors created by the user - their `grad_fn is None`).

If you want to compute the derivatives, you can call `.backward()` on a `Tensor`.  If `Tensor` is a scalar (i.e. t holds a one element data), you don't need to specify any arguments to `backward()`, however if it has more element, you need to specify a `gradient` argument that is a tensor of matching shape.



In [3]:
import torch

Create a tensor and set `requires_grad=True` to track computation with it:

In [4]:
x = torch.ones(2, 2, requires_grad=True)

Do a tensor operation:

In [5]:
y = x + 2
print(y)

tensor([[3., 3.],
        [3., 3.]], grad_fn=<AddBackward0>)


`y` was created as a result of an operation, so it has a `grad_fn`

In [6]:
print(y.grad_fn)

Do more operations on `y`

In [7]:
z = y * y * 3
out = z.mean()
print(z, out)

tensor([[27., 27.],
        [27., 27.]], grad_fn=<MulBackward0>) tensor(27., grad_fn=<MeanBackward0>)


`.requires_grad_()` changes an existing Tensor's `requires_grad` frag in-place.  The input flag defaults to `False` if not given.

In [8]:
a = torch.randn(2, 2)
a = ((a * 3) / (a - 1))
print(a.requires_grad)
a.requires_grad_(True)
print(a.requires_grad)
b = (a * a).sum()
print(b.grad_fn)

False
True


## Gradients

Let's backprop now.  Because `out` contains a single scalar, `out.backward()` is equivalent to `out.backward(torch.tensor(1.))`.


In [9]:
out.backward()

Print gradients d(out)/dx

In [10]:
print(x.grad)

tensor([[4.5000, 4.5000],
        [4.5000, 4.5000]])


You should have got a matrix of `4.5`.  Let's call the `out` Tensor "o".  We have that 

$o = \frac{1}{4}\sum_i z_i, z_i=3(x_i + 2)^2$

Therefore:

$\frac{\partial o}{\partial x_i} =
    \frac{3}{2}(x_i + 2)$

Hence:

$\frac{\partial o}{\partial x_i} 
    \Bigm\lvert_{x_i=1} = \frac{9}{2} = 4.5$

Mathematically, if you have a vector valued function $\mathbb{y} = f(\mathbb{x})$ then the gradient of $\mathbb{y}$ wrt. $\mathbb{x}$ is a Jacobian matrix:

$ J = 
\begin{pmatrix}
\frac{\partial y_1}{\partial x_1} & \cdots & \frac{\partial y_1}{\partial x_n} \\
\vdots & \ddots & \vdots \\
\frac{\partial y_m}{\partial x_1} & \cdots & \frac{\partial y_m}{\partial x_n}
\end{pmatrix}
$

Generally speaking, `torch.autograd` is an engine for computing vector-Jacobian products.   That is, given any vector $\mathbb{v} = \begin{pmatrix}v_1 & v_2 & \cdots & v_m\end{pmatrix}^T$, compute the product $\mathbb{v}^T \cdot J$.  If $\mathbb{v}$ happens to be the gradient of a scalar function $l = g(\mathbb{y})$, that is, $v = \begin{pmatrix} \frac{\partial l}{\partial y_1} && \cdots && \frac{\partial l}{\partial y_m} \end{pmatrix}^T$, then by the chain rule, the vector-Jacobian product would be the gradient of $l$ wrt. $\mathbb{x}$:

$
J^T \cdot \mathbb{v} = 
\begin{pmatrix}
\frac{\partial y_1}{\partial x_1} & \cdots & \frac{\partial y_m}{\partial x_1} \\
\vdots & \ddots & \vdots \\
\frac{\partial y_1}{\partial x_n} & \cdots & \frac{\partial y_m}{\partial x_n}
\end{pmatrix}
\begin{pmatrix}
\frac{\partial l}{\partial y_1} \\
\vdots \\
\frac{\partial l}{\partial y_m}
\end{pmatrix}
\begin{pmatrix}
\frac{\partial l}{\partial x_1} \\
\vdots \\
\frac{\partial l}{\partial x_n}
\end{pmatrix}
$


Note that $\mathbb{v}^T \cdot J$ produces a row vector which can be treated as a column vector by taking $J^T \cdot \mathbb{v}$.

This characteristic of vector-Jacobian product makes it very convenient to feed external gradients into a model that has non-scalar output.

Now let's take a look at an example of vector-Jacobian product:


In [11]:
x = torch.randn(3, requires_grad=True)

y = x * 2

while y.data.norm() < 1000:
    y = y * 2

print(y)

tensor([-946.0444, 1126.2666, -497.0979], grad_fn=<MulBackward0>)


Now in this case `y` is no longer a scalar.  `torch.autograd` could not compute the full Jacobian directly, but if we just want the vector-Jacobian product, simply pass the vector to `backward` as argument:

In [12]:
v = torch.tensor([0.1, 1.0, 0.0001], dtype=torch.float)
y.backward(v)
print(x.grad)

tensor([1.0240e+02, 1.0240e+03, 1.0240e-01])


You can also stop autograd from tracking history on Tenors with `.required_grad=True` by wrapping the code block in `with torch.no_grad:`

In [13]:
print(x.requires_grad)
print((x ** 2).requires_grad)

with torch.no_grad():
    print((x ** 2).requires_grad)

True
True
False


Alternatively you can use `.detach()` to get a new Tensor with the same content but hat does not require gradients:


In [14]:
print(x.requires_grad)
y = x.detach()
print(y.requires_grad)
print(x.eq(y).all())

True
False
tensor(True)
